## 1. Theoretical Foundation

### 1.1 The Two-Stage OCR Pipeline
Information extraction from images is a hierarchical process:
1. **Detection (Vision):** Identifying character regions using heatmaps (CRAFT architecture).
2. **Recognition (Sequence):** Mapping pixels to characters using CRNNs and **CTC Loss**.

### 1.2 Information Extraction (IE)
Once we have raw text, we use **Deterministic** methods (Regex) for structured data like emails, and **Probabilistic** methods (NLP/NER) for unstructured data like human names.

In [ ]:
!pip install "pydantic<2.0" easyocr spacy pillow --quiet
!python -m spacy download en_core_web_sm --quiet
!python -m spacy download de_core_news_sm --quiet

import easyocr
import spacy
import re
import numpy as np
import cv2
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
from IPython.display import display

## 2. Generating Synthetic Test Data
In this section, we create "Fake" business cards as images. This is a common practice in ML when real-world data is scarce or sensitive (GDPR).

In [ ]:
import os
from PIL import Image, ImageDraw, ImageFont
import requests
from io import BytesIO
from urllib.parse import urlparse

def create_hq_business_card(name, email, phone, filename="card_hq.png"):
    width, height = 1500, 875
    img = Image.new('RGB', (width, height), color=(255, 255, 255))
    d = ImageDraw.Draw(img)

    try:
        font_name = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", 80)
        font_info = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf", 50)
    except:
        font_name = ImageFont.load_default()
        font_info = ImageFont.load_default()
        print("Warning: Using default font. High-quality fonts not found.")

    d.rectangle([20, 20, width-20, height-20], outline=(40, 40, 40), width=10)
    d.line([100, 250, 600, 250], fill=(200, 0, 0), width=8)

    d.text((100, 150), name, fill=(0, 0, 0), font=font_name)
    d.text((100, 350), f"Telefon: {phone}", fill=(50, 50, 50), font=font_info)
    d.text((100, 430), f"E-Mail:  {email}", fill=(50, 50, 50), font=font_info)

    img.save(filename, quality=95, subsampling=0)
    return filename

def load_image_from_web(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content))
        return img
    except requests.RequestException as e:
        print(f"Error downloading image from {url}: {e}")
        return None

business_cards_data = [
    ("Mag. Thomas Müller", "t.mueller@tech.at", "+43 664 1234567", "card1_hq.png"),
    ("Dr. Ingrid Schmidt", "ingrid.s@firma.at", "01 555 9988", "card2_hq.png"),
    ("Ing. Christian Weber", "c.weber@solutions.at", "+43 1 234 5678", "card3_hq.png"),
    ("Dr. Prof. Anna Berger", "anna.berger@uni.at", "0664 987654", "card4_hq.png"),
    ("Mag. Stefan König", "s.koenig@consulting.at", "+43 2236 123456", "card5_hq.png"),
]

for name, email, phone, filename in business_cards_data:
    create_hq_business_card(name, email, phone, filename)

In [ ]:
!pip install faker --quiet
from faker import Faker
import re as regex

fake_at = Faker("de_AT")
fake_de = Faker("de_DE")

def generate_random_business_card(name, email, phone, filename="random_card.png", use_job=True):
    width, height = 1500, 875
    img = Image.new('RGB', (width, height), color=(255, 255, 255))
    d = ImageDraw.Draw(img)

    try:
        font_name = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", 70)
        font_job = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Italic.ttf", 45)
        font_info = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf", 45)
    except:
        font_name = ImageFont.load_default()
        font_job = ImageFont.load_default()
        font_info = ImageFont.load_default()

    d.rectangle([20, 20, width-20, height-20], outline=(40, 40, 40), width=8)
    d.line([100, 240, 600, 240], fill=(200, 0, 0), width=6)

    d.text((100, 120), name, fill=(0, 0, 0), font=font_name)

    if use_job:
        job = fake_at.job()
        d.text((100, 230), job, fill=(100, 100, 100), font=font_job)

    d.text((100, 330), f"Telefon: {phone}", fill=(50, 50, 50), font=font_info)
    d.text((100, 420), f"E-Mail:  {email}", fill=(50, 50, 50), font=font_info)

    img.save(filename, quality=95, subsampling=0)
    return filename

def extract_phone_from_text(phone_text):
    normalized = regex.sub(r'[\s\-\(\)/.]', '', phone_text)
    return normalized

def generate_card_with_labels(faker_instance, use_job=True):
    name = faker_instance.name()
    email = faker_instance.email()
    phone = faker_instance.phone_number()

    card_text = f"{name}\n"
    if use_job:
        card_text += f"{faker_instance.job()}\n"
    card_text += f"Telefon: {phone}\nE-Mail: {email}"

    ground_truth = {
        "Names": [name],
        "Emails": [email],
        "Phone Numbers": [phone]
    }

    return card_text, ground_truth

print("Generating random Austrian business cards\n")

dataset = []
for i in range(10):
    text, gt = generate_card_with_labels(fake_at, use_job=True)
    filename = f"eval_card_{i:02d}.png"

    generate_random_business_card(
        name=gt["Names"][0],
        email=gt["Emails"][0],
        phone=gt["Phone Numbers"][0],
        filename=filename,
        use_job=True
    )

    dataset.append((filename, gt))
    print(f"Generated {filename}")
    print(f"Name:  {gt['Names'][0]}")
    print(f"Email: {gt['Emails'][0]}")
    print(f"Phone: {gt['Phone Numbers'][0]}\n")

print(f"Generated {len(dataset)} random business cards for evaluation.")

## 3. The Austrian Information Extractor Model
This model uses a specialized Regex for the Austrian market and strips common academic titles to help the NLP model find names.

In [ ]:
from difflib import SequenceMatcher
import json

class UniversityContactModel:
    def __init__(self):
        self.reader = easyocr.Reader(['en', 'de'])
        self.nlp = spacy.load("de_core_news_sm")
        self.metrics = []

    def extract(self, img_path):
        results = self.reader.readtext(img_path, detail=0)
        if not results: return None, "No text found."

        full_text = " ".join(results)

        # clean_text = re.sub(r'\b(Mag\.|Dr\.|Ing\.|Dipl\.-Ing\.)\s?', '', full_text)

        email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'

        at_phone_pattern = r'(?:\+43|0043|0)(?:\s?\d{1,4})(?:[\s\-/]?\d{2,10})+'

        emails = list(set(re.findall(email_pattern, full_text)))
        phones = list(set(re.findall(at_phone_pattern, full_text)))

        noise_words = ["Tel", "Email", "Mobil", "Fax", "Mag", "Dr", "Ing"]
        doc = self.nlp(full_text)
        names = []
        for ent in doc.ents:
            if ent.label_ == "PER":
                clean_name = ent.text
                for word in noise_words:
                    clean_name = re.sub(rf'\b{word}\b', '', clean_name, flags=re.I).strip()
                if len(clean_name) > 2:
                    names.append(clean_name)

        return {
            "Names": list(set(names)),
            "Emails": [e.replace(" ", "") for e in emails],
            "Phone Numbers": phones
        }, full_text

    def calculate_similarity(self, detected, expected):
        if not detected or not expected:
            return 0.0
        return SequenceMatcher(None, detected.lower(), expected.lower()).ratio()

    def evaluate(self, predictions, ground_truth):
        metrics = {}

        pred_emails = set(predictions.get('Emails', []))
        true_emails = set(ground_truth.get('Emails', []))
        if true_emails:
            email_precision = len(pred_emails & true_emails) / len(pred_emails) if pred_emails else 0
            email_recall = len(pred_emails & true_emails) / len(true_emails)
            email_f1 = 2 * (email_precision * email_recall) / (email_precision + email_recall) if (email_precision + email_recall) > 0 else 0
            metrics['Email'] = {
                'Precision': round(email_precision, 3),
                'Recall': round(email_recall, 3),
                'F1-Score': round(email_f1, 3)
            }

        pred_phones = set(predictions.get('Phone Numbers', []))
        true_phones = set(ground_truth.get('Phone Numbers', []))
        if true_phones:
            phone_precision = len(pred_phones & true_phones) / len(pred_phones) if pred_phones else 0
            phone_recall = len(pred_phones & true_phones) / len(true_phones)
            phone_f1 = 2 * (phone_precision * phone_recall) / (phone_precision + phone_recall) if (phone_precision + phone_recall) > 0 else 0
            metrics['Phone'] = {
                'Precision': round(phone_precision, 3),
                'Recall': round(phone_recall, 3),
                'F1-Score': round(phone_f1, 3)
            }

        pred_names = predictions.get('Names', [])
        true_names = ground_truth.get('Names', [])
        if true_names:
            name_similarities = []
            for true_name in true_names:
                if pred_names:
                    max_sim = max([self.calculate_similarity(p, true_name) for p in pred_names])
                    name_similarities.append(max_sim)

            name_accuracy = sum(name_similarities) / len(true_names) if true_names else 0
            metrics['Name'] = {
                'Accuracy': round(name_accuracy, 3)
            }

        return metrics

model = UniversityContactModel()

## 4. Processing Results
We iterate through our generated images and display the extracted structured data.

In [ ]:
test_files = [
    ("card1_hq.png", {"Names": ["Thomas Müller"], "Emails": ["t.mueller@tech.at"], "Phone Numbers": ["+43 664 1234567"]}),
    ("card2_hq.png", {"Names": ["Ingrid Schmidt"], "Emails": ["ingrid.s@firma.at"], "Phone Numbers": ["01 555 9988"]}),
    ("card3_hq.png", {"Names": ["Christian Weber"], "Emails": ["c.weber@solutions.at"], "Phone Numbers": ["+43 1 234 5678"]}),
    ("card4_hq.png", {"Names": ["Anna Berger"], "Emails": ["anna.berger@uni.at"], "Phone Numbers": ["0664 987654"]}),
    ("card5_hq.png", {"Names": ["Stefan König"], "Emails": ["s.koenig@consulting.at"], "Phone Numbers": ["+43 2236 123456"]}),
]

results_summary = []

for file, ground_truth in test_files:
    data, raw = model.extract(file)

    display(Image.open(file))

    print(f"\n[RAW TEXT]: {raw}")
    if data:
        print("\n[EXTRACTED DATA]:")
        for key, value in data.items():
            val_str = ", ".join(value) if value else "None detected"
            print(f"  {key:15}: {val_str}")

        print("\n[GROUND TRUTH]:")
        for key, value in ground_truth.items():
            val_str = ", ".join(value) if value else "None"
            print(f"  {key:15}: {val_str}")

        metrics = model.evaluate(data, ground_truth)
        print("\n[ACCURACY METRICS]:")
        for field, scores in metrics.items():
            print(f"  {field}:")
            for metric, value in scores.items():
                print(f"    {metric}: {value}")

        results_summary.append({
            'File': file,
            'Metrics': metrics,
            'Prediction': data,
            'Ground_Truth': ground_truth
        })

print("All cards evaluated")

In [ ]:
print("PROCESSING GENERATED FAKER BUSINESS CARDS")

faker_results = []

for filename, ground_truth in dataset:
    if os.path.exists(filename):
        print(f"Processing: {filename}")

        predictions, raw_text = model.extract(filename)

        display(Image.open(filename))

        print(f"\nRaw OCR Text:\n{raw_text}\n")

        print("Extracted Data:")
        if predictions:
            for key, values in predictions.items():
                print(f"  {key:20}: {', '.join(values) if values else 'None detected'}")
        else:
            print("No data extracted")

        print("\nGround Truth:")
        for key, values in ground_truth.items():
            print(f"  {key:20}: {', '.join(values) if values else 'None'}")

        if predictions:
            metrics = model.evaluate(predictions, ground_truth)
            print("\nAccuracy Metrics:")
            for field, scores in metrics.items():
                print(f"  {field}:")
                for metric, value in scores.items():
                    print(f"    {metric:15}: {value}")

            faker_results.append({
                'File': filename,
                'Predictions': predictions,
                'Ground_Truth': ground_truth,
                'Metrics': metrics,
                'Raw_Text': raw_text
            })
        else:
            print("Could not evaluate - no predictions")

print("Finished Evaluation")

## 5. Visualization: Accuracy Metrics & Performance Analysis
Analyzing model performance across extracted fields (Name, Email, Phone) using precision, recall, and F1-scores.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

email_metrics = []
phone_metrics = []
name_metrics = []

for result in results_summary:
    metrics = result['Metrics']
    if 'Email' in metrics:
        email_metrics.append({
            'File': result['File'],
            'Precision': metrics['Email']['Precision'],
            'Recall': metrics['Email']['Recall'],
            'F1-Score': metrics['Email']['F1-Score']
        })

    if 'Phone' in metrics:
        phone_metrics.append({
            'File': result['File'],
            'Precision': metrics['Phone']['Precision'],
            'Recall': metrics['Phone']['Recall'],
            'F1-Score': metrics['Phone']['F1-Score']
        })

    if 'Name' in metrics:
        name_metrics.append({
            'File': result['File'],
            'Accuracy': metrics['Name']['Accuracy']
        })

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('OCR Model Performance Metrics on Synthetic Business Cards', fontsize=16, fontweight='bold')

if email_metrics:
    email_df = pd.DataFrame(email_metrics)
    x = range(len(email_df))
    width = 0.25
    axes[0, 0].bar([i - width for i in x], email_df['Precision'], width, label='Precision', alpha=0.8)
    axes[0, 0].bar([i for i in x], email_df['Recall'], width, label='Recall', alpha=0.8)
    axes[0, 0].bar([i + width for i in x], email_df['F1-Score'], width, label='F1-Score', alpha=0.8)
    axes[0, 0].set_xlabel('Business Card')
    axes[0, 0].set_ylabel('Score')
    axes[0, 0].set_title('Email Extraction Performance')
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels([f'Card {i+1}' for i in x], rotation=45)
    axes[0, 0].legend()
    axes[0, 0].set_ylim([0, 1.1])
    axes[0, 0].grid(axis='y', alpha=0.3)

if phone_metrics:
    phone_df = pd.DataFrame(phone_metrics)
    x = range(len(phone_df))
    axes[0, 1].bar([i - width for i in x], phone_df['Precision'], width, label='Precision', alpha=0.8)
    axes[0, 1].bar([i for i in x], phone_df['Recall'], width, label='Recall', alpha=0.8)
    axes[0, 1].bar([i + width for i in x], phone_df['F1-Score'], width, label='F1-Score', alpha=0.8)
    axes[0, 1].set_xlabel('Business Card')
    axes[0, 1].set_ylabel('Score')
    axes[0, 1].set_title('Phone Extraction Performance')
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels([f'Card {i+1}' for i in x], rotation=45)
    axes[0, 1].legend()
    axes[0, 1].set_ylim([0, 1.1])
    axes[0, 1].grid(axis='y', alpha=0.3)

if name_metrics:
    name_df = pd.DataFrame(name_metrics)
    colors = ['green' if acc >= 0.8 else 'orange' if acc >= 0.5 else 'red' for acc in name_df['Accuracy']]
    axes[1, 0].bar(range(len(name_df)), name_df['Accuracy'], color=colors, alpha=0.7)
    axes[1, 0].set_xlabel('Business Card')
    axes[1, 0].set_ylabel('Accuracy')
    axes[1, 0].set_title('Name Recognition Accuracy')
    axes[1, 0].set_xticks(range(len(name_df)))
    axes[1, 0].set_xticklabels([f'Card {i+1}' for i in range(len(name_df))], rotation=45)
    axes[1, 0].set_ylim([0, 1.1])
    axes[1, 0].axhline(y=0.8, color='r', linestyle='--', label='Good Threshold (0.8)', alpha=0.7)
    axes[1, 0].legend()
    axes[1, 0].grid(axis='y', alpha=0.3)

if email_metrics or phone_metrics:
    avg_data = {
        'Email F1': sum([m['F1-Score'] for m in email_metrics]) / len(email_metrics) if email_metrics else 0,
        'Phone F1': sum([m['F1-Score'] for m in phone_metrics]) / len(phone_metrics) if phone_metrics else 0,
        'Name Acc': sum([m['Accuracy'] for m in name_metrics]) / len(name_metrics) if name_metrics else 0,
    }
    metrics_names = list(avg_data.keys())
    metrics_values = list(avg_data.values())
    bars = axes[1, 1].bar(metrics_names, metrics_values, color=['#1f77b4', '#ff7f0e', '#2ca02c'], alpha=0.7)
    axes[1, 1].set_ylabel('Score')
    axes[1, 1].set_title('Average Performance Across All Cards')
    axes[1, 1].set_ylim([0, 1.1])
    axes[1, 1].grid(axis='y', alpha=0.3)

    for bar, value in zip(bars, metrics_values):
        height = bar.get_height()
        axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                       f'{value:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('ocr_performance_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

faker_email_metrics = []
faker_phone_metrics = []
faker_name_metrics = []

for result in faker_results:
    metrics = result['Metrics']
    filename = result['File'].replace('eval_card_', '').replace('.png', '')

    if 'Email' in metrics:
        faker_email_metrics.append({
            'Card': f"Card {filename}",
            'Precision': metrics['Email']['Precision'],
            'Recall': metrics['Email']['Recall'],
            'F1-Score': metrics['Email']['F1-Score']
        })

    if 'Phone' in metrics:
        faker_phone_metrics.append({
            'Card': f"Card {filename}",
            'Precision': metrics['Phone']['Precision'],
            'Recall': metrics['Phone']['Recall'],
            'F1-Score': metrics['Phone']['F1-Score']
        })

    if 'Name' in metrics:
        faker_name_metrics.append({
            'Card': f"Card {filename}",
            'Accuracy': metrics['Name']['Accuracy']
        })

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('OCR Model Performance on Faker-Generated Business Cards',
             fontsize=18, fontweight='bold', y=0.995)

if faker_email_metrics:
    email_df = pd.DataFrame(faker_email_metrics)
    x = range(len(email_df))
    width = 0.25

    axes[0, 0].bar([i - width for i in x], email_df['Precision'], width,
                    label='Precision', alpha=0.85, color='#1f77b4')
    axes[0, 0].bar([i for i in x], email_df['Recall'], width,
                    label='Recall', alpha=0.85, color='#ff7f0e')
    axes[0, 0].bar([i + width for i in x], email_df['F1-Score'], width,
                    label='F1-Score', alpha=0.85, color='#2ca02c')

    axes[0, 0].set_xlabel('Faker-Generated Card', fontsize=11, fontweight='bold')
    axes[0, 0].set_ylabel('Score', fontsize=11, fontweight='bold')
    axes[0, 0].set_title('Email Extraction Performance', fontsize=13, fontweight='bold')
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels([f'{i}' for i in range(len(email_df))], fontsize=9)
    axes[0, 0].legend(fontsize=10, loc='lower right')
    axes[0, 0].set_ylim([0, 1.15])
    axes[0, 0].grid(axis='y', alpha=0.3, linestyle='--')
    axes[0, 0].axhline(y=1.0, color='green', linestyle=':', linewidth=2, alpha=0.5)

if faker_phone_metrics:
    phone_df = pd.DataFrame(faker_phone_metrics)
    x = range(len(phone_df))
    width = 0.25

    axes[0, 1].bar([i - width for i in x], phone_df['Precision'], width,
                    label='Precision', alpha=0.85, color='#1f77b4')
    axes[0, 1].bar([i for i in x], phone_df['Recall'], width,
                    label='Recall', alpha=0.85, color='#ff7f0e')
    axes[0, 1].bar([i + width for i in x], phone_df['F1-Score'], width,
                    label='F1-Score', alpha=0.85, color='#2ca02c')

    axes[0, 1].set_xlabel('Faker-Generated Card', fontsize=11, fontweight='bold')
    axes[0, 1].set_ylabel('Score', fontsize=11, fontweight='bold')
    axes[0, 1].set_title('Phone Number Extraction Performance', fontsize=13, fontweight='bold')
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels([f'{i}' for i in range(len(phone_df))], fontsize=9)
    axes[0, 1].legend(fontsize=10, loc='lower right')
    axes[0, 1].set_ylim([0, 1.15])
    axes[0, 1].grid(axis='y', alpha=0.3, linestyle='--')
    axes[0, 1].axhline(y=1.0, color='green', linestyle=':', linewidth=2, alpha=0.5)

if faker_name_metrics:
    name_df = pd.DataFrame(faker_name_metrics)
    colors = ['#2ca02c' if acc >= 0.8 else '#ff7f0e' if acc >= 0.5 else '#d62728'
              for acc in name_df['Accuracy']]

    bars = axes[1, 0].bar(range(len(name_df)), name_df['Accuracy'],
                          color=colors, alpha=0.75, edgecolor='black', linewidth=1.5)
    axes[1, 0].set_xlabel('Faker-Generated Card', fontsize=11, fontweight='bold')
    axes[1, 0].set_ylabel('Accuracy', fontsize=11, fontweight='bold')
    axes[1, 0].set_title('Name Recognition Accuracy', fontsize=13, fontweight='bold')
    axes[1, 0].set_xticks(range(len(name_df)))
    axes[1, 0].set_xticklabels([f'{i}' for i in range(len(name_df))], fontsize=9)
    axes[1, 0].set_ylim([0, 1.15])
    axes[1, 0].axhline(y=0.8, color='red', linestyle='--', linewidth=2,
                       label='Good Threshold (0.8)', alpha=0.7)
    axes[1, 0].axhline(y=1.0, color='green', linestyle=':', linewidth=2, alpha=0.5)
    axes[1, 0].legend(fontsize=10, loc='lower right')
    axes[1, 0].grid(axis='y', alpha=0.3, linestyle='--')

    for bar in bars:
        height = bar.get_height()
        axes[1, 0].text(bar.get_x() + bar.get_width()/2., height + 0.02,
                       f'{height:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

if faker_email_metrics or faker_phone_metrics or faker_name_metrics:
    avg_data = {
        'Email\nF1-Score': sum([m['F1-Score'] for m in faker_email_metrics]) / len(faker_email_metrics) if faker_email_metrics else 0,
        'Phone\nF1-Score': sum([m['F1-Score'] for m in faker_phone_metrics]) / len(faker_phone_metrics) if faker_phone_metrics else 0,
        'Name\nAccuracy': sum([m['Accuracy'] for m in faker_name_metrics]) / len(faker_name_metrics) if faker_name_metrics else 0,
    }

    metrics_names = list(avg_data.keys())
    metrics_values = list(avg_data.values())
    colors_avg = ['#1f77b4', '#ff7f0e', '#2ca02c']

    bars = axes[1, 1].bar(metrics_names, metrics_values, color=colors_avg,
                          alpha=0.75, edgecolor='black', linewidth=2)
    axes[1, 1].set_ylabel('Average Score', fontsize=11, fontweight='bold')
    axes[1, 1].set_title('Average Performance Across All Cards',
                         fontsize=13, fontweight='bold')
    axes[1, 1].set_ylim([0, 1.15])
    axes[1, 1].grid(axis='y', alpha=0.3, linestyle='--')
    axes[1, 1].axhline(y=1.0, color='green', linestyle=':', linewidth=2, alpha=0.5)

    for bar, value in zip(bars, metrics_values):
        height = bar.get_height()
        axes[1, 1].text(bar.get_x() + bar.get_width()/2., height + 0.02,
                       f'{value:.3f}', ha='center', va='bottom',
                       fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('faker_ocr_performance_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Web Image Loading & Processing
Load business card images directly from URLs and extract information. Useful for testing OCR on real-world images without storing local copies.

In [ ]:

web_image_urls = [
    "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcR4X63sKw9jTPdqONtgYFkPlZBPJASGd8ghkg&s",
]

def process_web_image(url, expected_data=None):
    img = load_image_from_web(url)
    if img is None:
        print("Failed to load image")
        return None

    temp_filename = "temp_web_image.png"
    img.save(temp_filename)

    display(img)

    data, raw_text = model.extract(temp_filename)

    print(f"\n[RAW TEXT]: {raw_text}")
    if data:
        print("\n[EXTRACTED DATA]:")
        for key, value in data.items():
            val_str = ", ".join(value) if value else "None detected"
            print(f"  {key:15}: {val_str}")

        if expected_data:
            print("\n[GROUND TRUTH]:")
            for key, value in expected_data.items():
                val_str = ", ".join(value) if value else "None"
                print(f"  {key:15}: {val_str}")

            metrics = model.evaluate(data, expected_data)
            print("\n[ACCURACY METRICS]:")
            for field, scores in metrics.items():
                print(f"  {field}: {scores}")

    if os.path.exists(temp_filename):
        os.remove(temp_filename)

    return data

extracted_data = process_web_image(web_image_urls[0], {"Names": ["Name"], "Emails": ["Email"], "Phone Numbers": ["0123456789"]})